# 🎙️ Earnings Call Readiness Coach — Barenya Pvt. Ltd.

**Use case:** *Is the leadership team ready for tough investor questions?*

A listed company is preparing for its quarterly earnings call. Revenue has grown, but margins have fallen, cash flow has weakened and segments are performing unevenly. Investors will compare the results with earlier management commitments.

**This prototype helps leadership rehearse and improve their answers.** It follows a clear journey from inputs to a business recommendation:

| Step | What it does |
|---|---|
| ① Results | Shows the quarter's KPIs, trends, segments and margin bridge |
| ② Gap detector | Compares 9 earlier management promises with actual results and ranks the risk |
| ③ Likely questions | Turns the gaps into the tough questions analysts will ask |
| ④ Practise & score | Scores an executive's answer with Gemini AI plus rule-based disclosure guardrails |
| ⑤ Readiness | Gives a risk-weighted readiness verdict and recommendations for leadership |

**Tools:** Python, Streamlit (dashboard), Plotly (charts), Google Gemini (AI scoring).
**Data:** fully synthetic. Barenya Pvt. Ltd. is a fictional company; figures are in ₹ crore.

**How to run:** add your Gemini API key in Colab Secrets (🔑 icon) as `GEMINI_API_KEY`, then *Runtime → Run all*. The app link appears in Section 7.

## 0. Setup
Installs the libraries and creates a project folder. All app files are written into this folder using `%%writefile`.
⚠️ If Colab restarts, run this cell again first.

In [ ]:
!pip install -q streamlit google-genai plotly
import os
os.makedirs("/content/barenya_coach", exist_ok=True)
%cd /content/barenya_coach

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 62.5 MB/s eta 0:00:00
/content/barenya_coach


## 1. Company data (`data.py`)
Every input the app uses lives in one file, so the company can be swapped by editing just this file.

**Part A** holds the financial results (6 quarters), segment performance, the margin bridge, what management said on previous calls, and the approved disclosure pack: the only facts leadership may state on the call.

**Data is internally consistent:** segment revenues add up to the group total, FCF = operating cash flow − capex, and the margin bridge (−120 −100 −40 = −260 bps) explains the fall from 16.5% to 13.9%.

In [ ]:
%%writefile data.py
# data.py — every input the app uses. All figures are synthetic, in ₹ crore.
COMPANY_NAME = "Barenya Pvt. Ltd."

COMPANY = {
    "name": COMPANY_NAME,
    "ticker": "NSE: BARENYA (fictional)",
    "description": ("Fictional NSE-listed mid-cap with three segments: Digital Solutions (IT services), "
                    "Industrial Automation (machinery and controls) and Consumer Electronics Components "
                    "(parts for phone and appliance makers)."),
    "quarter": "Q2 FY27 (Jul–Sep 2026)",
    "call_date": "October 2026",
}

# Six quarters of results. FCF = operating cash flow (cfo) − capex.
QUARTERLY = {
    "quarter":         ["Q1 FY26", "Q2 FY26", "Q3 FY26", "Q4 FY26", "Q1 FY27", "Q2 FY27"],
    "revenue":         [2360, 2450, 2520, 2610, 2640, 2793],
    "ebitda":          [396,  404,  408,  412,  396,  388],
    "pat":             [238,  246,  244,  241,  205,  189],
    "cfo":             [380,  330,  310,  300,  190,  85],
    "capex":           [150,  120,  130,  150,  160,  175],
    "net_debt":        [None, None, None, 610,  700,  840],
    "receivable_days": [60,   62,   63,   64,   72,   78],
}

# Segment revenue (₹ cr) and EBIT margin (%): same quarter last year, last quarter, this quarter
SEGMENTS = {
    "segment":       ["Digital Solutions", "Industrial Automation", "Consumer Electronics Components"],
    "rev_q2fy26":    [900,  1000, 550],
    "rev_q1fy27":    [1090, 1040, 510],
    "rev_q2fy27":    [1188, 1099, 506],
    "margin_q2fy26": [14.0, 15.5, 9.0],
    "margin_q1fy27": [10.5, 15.4, 4.5],
    "margin_q2fy27": [8.5,  15.1, 2.0],
}

# Why EBITDA margin fell from 16.5% to 13.9% (in basis points)
MARGIN_BRIDGE = [
    ("Q2 FY26 margin", 16.5, "absolute"),
    ("Digital deal transition costs", -1.2, "relative"),
    ("Consumer under-utilisation", -1.0, "relative"),
    ("Industrial input costs", -0.4, "relative"),
    ("Q2 FY27 margin", 13.9, "total"),
]

# What management said on earlier calls (investors WILL compare against these)
TRANSCRIPTS = [
    ("T1", "Q4 FY26 call (May 2026)", "CEO", "We are confident FY27 will be a year of margin expansion. We expect EBITDA margins to improve by 50 to 100 basis points from FY26's 16.3% as our pricing actions flow through."),
    ("T2", "Q4 FY26 call (May 2026)", "CFO", "Free cash flow remains a priority. We expect to be FCF positive in every quarter of FY27 and to deliver over ₹600 crore of free cash flow for the year."),
    ("T3", "Q4 FY26 call (May 2026)", "CEO", "Consumer has bottomed out. With two new OEM wins, we expect the segment to return to growth by Q2."),
    ("T4", "Q4 FY26 call (May 2026)", "CFO", "We will keep net debt below ₹700 crore. FY27 capex will be ₹550 to 600 crore, and revenue growth 12 to 14%."),
    ("T5", "Q4 FY26 call (May 2026)", "Head of Digital", "Digital Solutions should grow 25% plus at mid-teens EBIT margins of 14 to 16%."),
    ("T6", "Q1 FY27 call (Aug 2026)", "CFO", "The Q1 margin dip is temporary and linked to the annual wage hike. We remain comfortable with our full-year margin guidance."),
    ("T7", "Q1 FY27 call (Aug 2026)", "CFO", "Working capital was seasonally higher. We expect receivable days to normalise back to around 62 in Q2."),
    ("T8", "Q1 FY27 call (Aug 2026)", "CEO", "Digital is winning large deals. Some carry upfront transition costs, but they will be margin-accretive by H2."),
]

# The approved Q2 FY27 disclosure pack = the ONLY facts leadership may state on the call
DISCLOSURES = [
    "Q2 FY27 revenue grew 14.0% YoY to ₹2,793 crore; H1 FY27 revenue grew 13.0% to ₹5,433 crore.",
    "Q2 FY27 EBITDA was ₹388 crore; EBITDA margin 13.9%, down 260 bps YoY from 16.5%. H1 FY27 EBITDA margin was 14.4%.",
    "Margin bridge: about 120 bps from transition costs on three large Digital deals, about 100 bps from Consumer under-utilisation, about 40 bps from Industrial input costs not yet passed through.",
    "PAT was ₹189 crore, down 23% YoY from ₹246 crore, also reflecting higher depreciation (₹85 crore) and finance cost (₹38 crore).",
    "Digital Solutions revenue grew 32% to ₹1,188 crore; EBIT margin 8.5% vs 14.0%. Transition costs end by Q4 FY27; Digital margin expected at 12–13% by Q4. Mid-teens remains the medium-term ambition (no timeline). Large deals expected to be margin-accretive over contract life.",
    "Industrial Automation revenue grew 9.9% to ₹1,099 crore; EBIT margin 15.1% vs 15.5%. Renegotiating input-cost pass-through clauses with key customers.",
    "Consumer Electronics Components revenue fell 8.0% to ₹506 crore; EBIT margin 2.0%. First OEM programme began supplies in Q2 at low volumes; the second was delayed by the customer's launch postponement and is now expected to start in Q4 FY27.",
    "Operating cash flow ₹85 crore; capex ₹175 crore; free cash flow negative ₹90 crore. H1 FY27 FCF negative ₹60 crore.",
    "Receivable days rose to 78 (62 a year ago) due to milestone-based billing on large Digital contracts. Standard credit terms unchanged. About ₹180 crore of milestone billing is due in Q3.",
    "Net debt ₹840 crore (₹610 crore in March 2026), about 0.52x TTM EBITDA. Expected to peak around ₹850 crore and return below ₹700 crore by March 2027.",
    "H1 capex of ₹335 crore was front-loaded for a new automation line at the Industrial plant. FY27 capex plan trimmed to ₹500–550 crore (H2 capex ₹165–215 crore).",
    "Project Sankalp cost programme: ₹90 crore annualised savings, full run-rate from Q4 FY27.",
    "Revised FY27 outlook: revenue growth 12–14% retained; EBITDA margin revised to 14.5–15.5% (from 16.8–17.3%); FCF positive in H2 with FY27 FCF of ₹150–250 crore. Progress reported every quarter.",
    "Dividend policy (30% payout) unchanged.",
]
# Other numbers it is fine to say (derived figures, e.g. TTM EBITDA, H1 capex run-rate)
EXTRA_ALLOWED_NUMBERS = [0.52, 12.95, 14.43, 16.3, 16.8, 17.3, 1604, 4810, 670, 170, 230, 850, 60]

Writing data.py


**Part B** links each earlier promise to the actual result, and adds the investor question bank, demo answers and assumptions.

For each promise we record: the committed range, the actual, which direction is good, a *scale* (how big a miss counts as maximum severity) and *investor sensitivity* (1–5, how much investors care).

In [ ]:
%%writefile -a data.py

# ---------- Part B: commitments vs actuals, questions, assumptions ----------
# For each earlier promise: the committed range (low/high), the actual, which direction is good,
# a "scale" (how big a miss counts as maximum severity) and investor sensitivity (1–5).
GAPS = [
    dict(id="margin", topic="EBITDA margin vs guidance", commitment="FY27 EBITDA margin up 50–100 bps to 16.8–17.3%", source="T1, T6",
         actual=14.43, low=16.8, high=17.3, good="higher", unit="%", scale=2.0, sensitivity=5, owner="CFO",
         key_fact="Revised margin outlook 14.5–15.5%; 260 bps bridge (120 Digital / 100 Consumer / 40 Industrial)"),
    dict(id="fcf", topic="Free cash flow", commitment="FCF positive every quarter; FY27 FCF > ₹600 cr", source="T2",
         actual=-90, low=0, high=None, good="higher", unit="₹ cr", scale=100, sensitivity=5, owner="CFO",
         key_fact="FCF positive in H2; FY27 FCF ₹150–250 cr; ₹180 cr milestone billing due in Q3"),
    dict(id="digital", topic="Digital segment margin", commitment="Digital EBIT margin 14–16%", source="T5, T8",
         actual=8.5, low=14, high=16, good="higher", unit="%", scale=4, sensitivity=4, owner="Head of Digital",
         key_fact="Transition costs end by Q4; Digital margin 12–13% by Q4"),
    dict(id="receivables", topic="Working capital / receivables", commitment="Receivable days back to ~62 in Q2", source="T7",
         actual=78, low=None, high=62, good="lower", unit="days", scale=20, sensitivity=3, owner="CFO",
         key_fact="Milestone billing on Digital deals; credit terms unchanged; ₹180 cr due in Q3"),
    dict(id="consumer", topic="Consumer segment turnaround", commitment="Consumer back to YoY growth by Q2", source="T3",
         actual=-8.0, low=0, high=None, good="higher", unit="%", scale=10, sensitivity=3, owner="CEO",
         key_fact="First OEM programme live; second delayed to Q4 FY27 (customer launch postponed)"),
    dict(id="debt", topic="Net debt and balance sheet", commitment="Net debt below ₹700 cr", source="T4",
         actual=840, low=None, high=700, good="lower", unit="₹ cr", scale=200, sensitivity=3, owner="CFO",
         key_fact="0.52x net debt/EBITDA; peak ~₹850 cr; below ₹700 cr by Mar-27; dividend policy unchanged"),
    dict(id="capex", topic="Capex run-rate", commitment="FY27 capex ₹550–600 cr", source="T4",
         actual=670, low=None, high=600, good="lower", unit="₹ cr", scale=150, sensitivity=2, owner="CFO",
         key_fact="H1 front-loaded (₹335 cr); FY27 plan trimmed to ₹500–550 cr"),
    dict(id="revenue", topic="Revenue growth", commitment="FY27 revenue growth 12–14%", source="T4",
         actual=12.95, low=12, high=14, good="higher", unit="%", scale=4, sensitivity=3, owner="CEO",
         key_fact="H1 growth 13.0%, in line with guidance; Digital +32%, Industrial +9.9%"),
    dict(id="digital_growth", topic="Digital growth", commitment="Digital growth 25%+", source="T5",
         actual=32.0, low=25, high=None, good="higher", unit="%", scale=10, sensitivity=2, owner="Head of Digital",
         key_fact="Digital +32% YoY, ahead of the 25%+ target"),
]

# Old promises that the revised outlook replaces. If an answer repeats one AS IF STILL VALID, flag it.
SUPERSEDED = [
    ("Margin expansion of 50–100 bps / 16.8–17.3% margin", ["margin expansion", "50 to 100 basis points", "50–100 bps", "50-100 bps", "17%", "17.3", "16.8"]),
    ("FCF positive every quarter / FY27 FCF over ₹600 crore", ["positive every quarter", "positive in every quarter", "600 crore", "₹600"]),
    ("Consumer back to growth by Q2", ["bottomed out", "growth by q2"]),
    ("Margin dip is temporary / comfortable with full-year margin guidance", ["dip is temporary", "comfortable with our full-year", "maintain our margin guidance"]),
    ("Net debt will stay below ₹700 crore this year", ["stay below 700", "remain below 700", "never exceed"]),
    ("Capex of ₹550–600 crore", ["550 to 600", "550–600", "550-600"]),
]

QUESTIONS = [
    dict(id="Q1", gap="margin", persona="Sell-side analyst (domestic brokerage)",
         question="In May you guided to 50–100 bps of EBITDA margin expansion for FY27, and in August you said you were still comfortable with it. H1 margin is 14.4%, more than 200 bps below even the low end. Is the guidance withdrawn, and why should we trust the new number?",
         why="Compares results with two earlier statements (T1, T6). A credibility question.",
         keywords=["margin", "guidance", "bps", "revis"],
         model_answer="You're right. H1 margin of 14.4% is well below the 50–100 bps expansion we guided to in May, and today we are revising our FY27 EBITDA margin outlook to 14.5–15.5%. Three factors explain the 260 bps year-on-year decline in Q2: about 120 bps from transition costs on three large Digital deals, about 100 bps from under-utilisation in Consumer, and about 40 bps from Industrial input costs not yet passed through. The Digital transition costs end by Q4, and Project Sankalp should deliver ₹90 crore of annualised savings at full run-rate from Q4. We would rather reset now to a number we can deliver than defend the old one.",
         follow_up="If transition costs end in Q4, why isn't the revised range higher? What are you assuming for Consumer?"),
    dict(id="Q2", gap="margin", persona="Institutional investor (long-only fund)",
         question="Can you break down the 260 bps year-on-year margin decline? How much is one-off and how much is structural?",
         why="Investors need to know if margin pressure carries into next year's valuation.",
         keywords=["bps", "one-off", "structural", "transition", "consumer", "input"],
         model_answer="Of the 260 bps decline, about 120 bps comes from upfront transition costs on three large Digital deals. That is time-bound and ends by Q4, after which we expect Digital margins of 12–13%. About 100 bps is Consumer under-utilisation, which should reverse as the second OEM programme starts supplies in Q4. The 40 bps from Industrial input costs is the most structural piece, and we are renegotiating pass-through clauses with key customers. So most of the decline is temporary, but we are not assuming a full recovery this year, which is why the revised outlook is 14.5–15.5%.",
         follow_up="What happens to the 40 bps if customers refuse the new pass-through terms?"),
    dict(id="Q3", gap="fcf", persona="Credit analyst (bond investor)",
         question="You committed to positive free cash flow in every quarter of FY27. Q2 FCF is negative ₹90 crore and H1 is negative ₹60 crore. What went wrong, and when does cash turn?",
         why="A broken cash promise (T2) hits credibility and the credit view.",
         keywords=["cash", "fcf", "working capital", "billing", "h2"],
         model_answer="We did not meet our commitment of positive free cash flow every quarter. Q2 FCF was negative ₹90 crore. The main driver is working capital: our large Digital contracts are billed on milestones, so receivable days rose to 78, and about ₹180 crore of milestone billing falls due in Q3. Capex was also front-loaded at ₹335 crore in H1, and we have trimmed the full-year capex plan to ₹500–550 crore. We now expect FCF to turn positive in H2, with FY27 FCF of ₹150–250 crore, and net debt to return below ₹700 crore by March 2027.",
         follow_up="What happens to your FCF outlook if the ₹180 crore milestone billing slips into Q4?"),
    dict(id="Q4", gap="receivables", persona="Sell-side analyst (foreign brokerage)",
         question="Receivable days went from 62 to 78, even though you told us in Q1 they would normalise in Q2. Are you financing customers to buy revenue growth?",
         why="Hints at aggressive revenue recognition; contradicts T7.",
         keywords=["receivable", "days", "billing", "credit"],
         model_answer="That's a fair challenge. Receivable days rose from 62 to 78, more than the normalisation we indicated in Q1. This is not extended credit: our standard customer credit terms are unchanged. The increase comes from milestone-based billing on our large Digital contracts, where work is delivered ahead of the billing milestone. About ₹180 crore of milestone billing is due in Q3, which should bring receivables down meaningfully.",
         follow_up="Can you show unbilled revenue separately from trade receivables?"),
    dict(id="Q5", gap="digital", persona="Institutional investor (long-only fund)",
         question="Digital grew 32% but its EBIT margin almost halved to 8.5%. Are you buying large deals at the expense of profitability?",
         why="Growth vs profitability; contradicts the mid-teens margin promise (T5).",
         keywords=["digital", "margin", "deal", "transition"],
         model_answer="Digital revenue grew 32%, ahead of our 25%-plus target, but EBIT margin fell to 8.5% from 14.0%. That is mainly upfront transition costs on three large multi-year deals, which we absorb in the early quarters. These costs end by Q4, and we expect Digital margins to recover to 12–13% by then. Mid-teens remains our medium-term ambition, but we won't put a date on it today. These deals are expected to be margin-accretive over the life of the contracts.",
         follow_up="How many more large deals in the pipeline carry similar transition costs?"),
    dict(id="Q6", gap="consumer", persona="Sell-side analyst (domestic brokerage)",
         question="You said Consumer had bottomed out and would return to growth by Q2. It's down 8%. What happened to the two OEM wins?",
         why="Direct reversal of T3.",
         keywords=["consumer", "oem", "growth", "delay"],
         model_answer="We expected Consumer to return to growth by Q2, and it didn't: revenue was down 8%. Of the two OEM wins we discussed, the first began supplies this quarter at low volumes. The second was delayed because the customer postponed its product launch, and we now expect supplies to start in Q4 FY27. Under-utilisation in the meantime cost us about 100 bps of group margin this quarter.",
         follow_up="Is Consumer still a core business, or are you considering strategic options?"),
    dict(id="Q7", gap="debt", persona="Credit analyst (bond investor)",
         question="Net debt is ₹840 crore against your ₹700 crore ceiling. Is the balance sheet at risk, and is the dividend safe?",
         why="Breaks the T4 promise; dividend matters to income investors.",
         keywords=["net debt", "leverage", "dividend", "700"],
         model_answer="Net debt rose to ₹840 crore, above the ₹700 crore level we indicated, mainly because of the working-capital build and front-loaded capex. Leverage remains comfortable at about 0.52x net debt to trailing EBITDA. We expect net debt to peak around ₹850 crore and return below ₹700 crore by March 2027 as milestone billing converts to cash. Our dividend policy of a 30% payout is unchanged.",
         follow_up="What is the covenant headroom on your term loans?"),
    dict(id="Q8", gap="margin", persona="Activist-leaning shareholder",
         question="This is the second quarter in a row where management has called an issue temporary. Why should investors believe your revised guidance?",
         why="Attacks the pattern of misses (T6, T7, T8).",
         keywords=["guidance", "credib", "revised", "believe", "deliver"],
         model_answer="That's a fair concern, and it's why we are resetting guidance today rather than defending it. We were too optimistic about how quickly Digital transition costs and the Consumer OEM ramp would play out. The revised outlook, 14.5–15.5% EBITDA margin and ₹150–250 crore of FCF for FY27, is based on contracts already signed and cost actions already under way, including ₹90 crore of annualised savings from Project Sankalp. We will report progress against each of these every quarter.",
         follow_up="Will management pay be linked to delivering the revised outlook?"),
    dict(id="Q9", gap="revenue", persona="Sell-side analyst (foreign brokerage)",
         question="Revenue is tracking guidance, but how much of that growth is low-margin large deals? Are you sacrificing quality of revenue?",
         why="Even good news gets challenged; a chance to lead with strengths.",
         keywords=["revenue", "growth", "quality", "deal"],
         model_answer="H1 revenue grew 13.0%, within our 12–14% guidance, and Industrial grew 9.9% with margins broadly stable at 15.1%. In Digital, the three large deals do dilute margins in the early quarters because of transition costs, but they are multi-year contracts expected to be margin-accretive over their life. Once transition costs end in Q4, we expect Digital margins of 12–13%. So we are not chasing volume at any price; we are absorbing cost upfront on revenue that is contracted.",
         follow_up="What share of H2 revenue is already contracted?"),
    dict(id="Q10", gap="capex", persona="Institutional investor (long-only fund)",
         question="You've spent ₹335 crore of capex in H1 against a full-year guide of ₹550–600 crore. Will capex overshoot and make the cash position worse?",
         why="Links capex discipline to the cash miss.",
         keywords=["capex", "h2", "plan", "cash"],
         model_answer="H1 capex was ₹335 crore because spending on a new automation line at our Industrial plant was front-loaded. We have reviewed the H2 programme and trimmed FY27 capex to ₹500–550 crore, below the earlier plan, so H2 capex will be ₹165–215 crore. That supports our expectation of positive free cash flow in H2.",
         follow_up="Does trimming capex delay any growth projects?"),
]

DEMO_ANSWERS = {
    "weak": "Look, we remain very confident in our business. The margin dip is temporary and we will definitely get back to 17% margins by year end, no doubt about it. Our digital business is doing fantastic and we are winning everywhere. I don't think there is any need to worry.",
    "average": "Margins were lower this quarter mainly because of transition costs on some large Digital deals and weakness in Consumer. These are temporary and we expect margins to improve in the second half as these costs roll off. We are taking cost actions and we remain confident in the long-term margin profile of the business.",
}

ASSUMPTIONS = [
    f"{COMPANY_NAME} is fictional. All figures are synthetic, in ₹ crore, built to match the case: revenue up, margins down, cash weaker, segments mixed.",
    "The call being prepared is the Q2 FY27 call (Jul–Sep 2026 quarter) in October 2026. Earlier promises come from the Q4 FY26 call (May 2026) and Q1 FY27 call (Aug 2026).",
    "The Q2 disclosure pack is the board-approved source of truth. Any figure said on the call that is not in it is flagged as unverified.",
    "FCF = operating cash flow − capex. Net debt/EBITDA uses trailing-12-month EBITDA (₹1,604 crore). Net debt rose ₹230 crore in H1: −₹60 crore FCF plus a ~₹170 crore final dividend.",
    "Gap severity (0–5) = how far the actual is outside the promised range ÷ a metric-specific scale (e.g. 2 percentage points for margin, ₹100 crore for FCF), capped at 5.",
    "Investor sensitivity (1–5) is a judgement: margins and cash weighted highest, capex lowest.",
    "Risk = severity × sensitivity (0–25). Red ≥ 15, Amber 7–15, Green < 7.",
    "Answer score (0–100) = Accuracy 25% + Consistency 25% + Directness 20% + Forward-looking discipline 15% + Clarity 15%.",
    "Gemini judges answer quality; fixed rules (unverified numbers, repeated old promises, absolute promises, evasive phrases) cap the AI's score where they apply.",
    "Readiness index = risk-weighted average of each topic's best rehearsal score. Ready ≥ 75, Nearly ready 55–75, Not ready < 55. Unrehearsed topics count as 0.",
]

Appending to data.py


**Check:** the table below should show revenue rising (2,360 → 2,793) while EBITDA margin falls (16.8% → 13.9%), free cash flow turns negative (230 → −90) and net debt climbs to 840. That is the case brief, expressed in numbers.

In [ ]:
import importlib, pandas as pd
import data
importlib.reload(data)   # picks up any edits you make to data.py

q = pd.DataFrame(data.QUARTERLY)
q["fcf"] = q.cfo - q.capex
q["ebitda_margin_%"] = (q.ebitda / q.revenue * 100).round(1)
print(data.COMPANY_NAME, "|", len(data.GAPS), "promises tracked |", len(data.QUESTIONS), "questions")
q[["quarter", "revenue", "ebitda_margin_%", "fcf", "net_debt"]]

Barenya Pvt. Ltd. | 9 promises tracked | 10 questions


,quarter,revenue,ebitda_margin_%,fcf,net_debt
0,Q1 FY26,2360,16.8,230,NaN
1,Q2 FY26,2450,16.5,210,NaN
2,Q3 FY26,2520,16.2,180,NaN
3,Q4 FY26,2610,15.8,150,610.0
4,Q1 FY27,2640,15.0,30,700.0
5,Q2 FY27,2793,13.9,-90,840.0


## 2. Gap detector (`engine.py`, part 1)
Checks every earlier promise against the actual result.

- **Severity (0–5)** = how far the actual is outside the promised range ÷ scale, capped at 5
- **Risk (0–25)** = severity × investor sensitivity
- **RAG:** Red ≥ 15 · Amber 7–15 · Green < 7

*Example:* management promised an EBITDA margin of at least 16.8%; the actual is 14.43%, a miss of 2.37 points. With a scale of 2 points, severity = 5 × 2.37 ÷ 2 = 5.9 → capped at 5. Sensitivity is 5, so **risk = 25 (the maximum)**.

In [ ]:
%%writefile engine.py
# engine.py — Part 1: gap detector (deterministic, no AI)
import re
import data as D


def score_gap(g):
    """Severity 0–5 = how far the actual is outside the promised range ÷ scale. Risk = severity × sensitivity."""
    a, miss = g["actual"], 0.0
    if g["good"] == "higher" and g["low"] is not None and a < g["low"]:
        miss = g["low"] - a
    if g["good"] == "lower" and g["high"] is not None and a > g["high"]:
        miss = a - g["high"]
    severity = min(5.0, 5.0 * miss / g["scale"])
    risk = round(severity * g["sensitivity"], 1)
    rag = "Red" if risk >= 15 else "Amber" if risk >= 7 else "Green"
    return {**g, "severity": round(severity, 1), "risk": risk, "rag": rag,
            "status": "Missed" if miss > 0 else "Met / on track"}


def detect_gaps():
    return sorted([score_gap(g) for g in D.GAPS], key=lambda g: -g["risk"])

Writing engine.py


**Check:** margin (25), free cash flow (22.5) and Digital margin (20) should be **Red**; receivables, Consumer and net debt **Amber**; capex, revenue and Digital growth **Green**. Revenue growth (12.95%) is within the 12–14% guidance, which is good news leadership should open with.

In [ ]:
import engine
importlib.reload(engine)
pd.DataFrame(engine.detect_gaps())[["topic", "commitment", "actual", "severity", "sensitivity", "risk", "rag"]]

,topic,commitment,actual,severity,sensitivity,risk,rag
0,EBITDA margin vs guidance,FY27 EBITDA margin up 50–100 bps to 16.8–17.3%,14.43,5.0,5,25.0,Red
1,Free cash flow,FCF positive every quarter; FY27 FCF > ₹600 cr,-90.00,4.5,5,22.5,Red
2,Digital segment margin,Digital EBIT margin 14–16%,8.50,5.0,4,20.0,Red
3,Working capital / receivables,Receivable days back to ~62 in Q2,78.00,4.0,3,12.0,Amber
4,Consumer segment turnaround,Consumer back to YoY growth by Q2,-8.00,4.0,3,12.0,Amber
5,Net debt and balance sheet,Net debt below ₹700 cr,840.00,3.5,3,10.5,Amber
6,Capex run-rate,FY27 capex ₹550–600 cr,670.00,2.3,2,4.7,Green
7,Revenue growth,FY27 revenue growth 12–14%,12.95,0.0,3,0.0,Green
8,Digital growth,Digital growth 25%+,32.00,0.0,2,0.0,Green


## 3. Answer guardrails (`engine.py`, part 2)
Fixed rules that check a practice answer the way a compliance officer would, on five dimensions:

| Dimension | Weight | Rule |
|---|---|---|
| Factual accuracy | 25% | Every figure must appear in the approved disclosure pack |
| Consistency | 25% | Must not repeat a replaced promise *as if it still stands*. Mentioning it to admit the miss is fine |
| Directness | 20% | Covers the question's topic and acknowledges the gap; no evasive phrases |
| Forward-looking discipline | 15% | No absolute promises ("definitely", "guarantee") |
| Clarity | 15% | 60–170 words (≈30–75 seconds spoken), no filler words |

`apply_guardrails` is the key design choice: when Gemini scores an answer, **the AI can never score higher than the rules allow** on accuracy, consistency and forward-looking discipline. A fluent answer with a made-up figure still fails.

In [ ]:
%%writefile -a engine.py


# engine.py — Part 2: answer guardrails (rule-based checks on a practice answer)
WEIGHTS = {"accuracy": 25, "consistency": 25, "directness": 20, "forward": 15, "clarity": 15}
LABELS = {"accuracy": "Factual accuracy", "consistency": "Consistency with past statements",
          "directness": "Directness", "forward": "Forward-looking discipline", "clarity": "Clarity and length"}

ABSOLUTE = ["definitely", "guarantee", "certainly", "surely", "no doubt", "100%", "for sure", "absolutely", "promise"]
EVASIVE = ["too early to say", "won't comment", "can't comment", "no need to worry", "nothing to worry",
           "i don't think there is any", "doing fantastic", "winning everywhere"]
ACKNOWLEDGE = ["miss", "below", "lower than", "fell", "did not meet", "didn't", "we expected", "you're right",
               "fair", "revis", "reset", "above the", "more than the", "declin", "dilute"]
PAST_REF = ["did not", "didn't", "miss", "guided", "we expected", "indicated", "earlier", "in may",
            "committed", "commitment", "below the", "above the", "revis", "reset", "replac", "no longer"]
FILLER = ["look", "basically", "you know", "frankly", "honestly", "kind of", "sort of"]


def numbers_in(text, skip_periods=True):
    """Pull figures out of text. Skips periods like Q3, H2, FY27 and calendar years."""
    found = []
    for m in re.finditer(r"(?<![\d.,])(FY|Q|H)?(\d[\d,]*\.?\d*)", text):
        if m.group(1) and skip_periods:
            continue
        raw = m.group(2).replace(",", "").rstrip(".")
        v = float(raw)
        if not (2000 <= v <= 2100 and "." not in raw):
            found.append(v)
    return found


def approved_numbers():
    nums = set(D.EXTRA_ALLOWED_NUMBERS) | {1, 2, 3, 4}
    for text in D.DISCLOSURES + [t[3] for t in D.TRANSCRIPTS]:
        nums.update(numbers_in(text))
    for table in (D.QUARTERLY, D.SEGMENTS):
        for col in table.values():
            nums.update(v for v in col if isinstance(v, (int, float)))
    return nums


def is_approved(v, approved):
    return any(abs(v - a) <= max(0.06, abs(a) * 0.005) for a in approved)


def rule_check(answer, q):
    low = answer.lower()
    words = len(answer.split())
    flags = []   # (level, message): level is "error", "warn" or "ok"

    # 1. Accuracy: every figure must be in the disclosure pack
    nums = numbers_in(answer)
    approved = approved_numbers()
    bad = sorted({n for n in nums if not is_approved(n, approved)})
    accuracy = 10 - 3 * len(bad)
    if not nums:
        accuracy = min(accuracy, 5)
        flags.append(("warn", "No figures cited. Investors expect numbers from the disclosure pack."))
    for n in bad:
        flags.append(("error", f"Figure '{n:g}' is not in the approved disclosure pack."))

    # 2. Consistency: do not repeat an old promise as if it still stands
    consistency = 10
    sentences = [s.lower() for s in re.split(r"(?<=[.!?])\s+", answer) if s.strip()]
    for commitment, patterns in D.SUPERSEDED:
        for s in sentences:
            hit = next((p for p in patterns if p in s), None)
            if hit and any(w in s for w in PAST_REF):
                flags.append(("ok", f"Refers to the old promise ('{commitment}') as replaced. Good."))
                break
            if hit:
                consistency -= 4
                flags.append(("error", f"Repeats a replaced promise as if still valid: '{commitment}'."))
                break

    # 3. Directness: covers the question's topic and admits the gap
    covered = sum(k in low for k in q["keywords"]) / len(q["keywords"])
    admits = any(a in low for a in ACKNOWLEDGE)
    directness = 6 * covered + (4 if admits else 0)
    if not admits:
        flags.append(("warn", "Does not acknowledge the gap versus earlier guidance. That sounds evasive."))
    for e in [e for e in EVASIVE if e in low]:
        directness -= 2
        flags.append(("error", f"Evasive or dismissive phrase: '{e}'."))

    # 4. Forward-looking discipline: no absolute promises
    forward = 10
    for a in [a for a in ABSOLUTE if a in low]:
        forward -= 3
        flags.append(("error", f"Absolute promise '{a}'. Say 'we expect' and use the disclosed outlook."))

    # 5. Clarity: 60–170 words is about 30–75 seconds spoken
    clarity = 10 if 60 <= words <= 170 else 7 if 35 <= words <= 230 else 4
    if clarity < 10:
        flags.append(("warn", f"{words} words. Aim for 60–170 (about 30–75 seconds spoken)."))
    for f in [f for f in FILLER if re.search(rf"\b{f}\b", low)]:
        clarity -= 1
        flags.append(("warn", f"Filler word '{f}'."))

    scores = {k: round(max(0, min(10, v)), 1) for k, v in dict(
        accuracy=accuracy, consistency=consistency, directness=directness, forward=forward, clarity=clarity).items()}
    if not any(level != "ok" for level, _ in flags):
        flags.append(("ok", "No guardrail issues found."))
    return {"scores": scores, "overall": overall(scores), "flags": flags}


def overall(scores):
    return round(sum(scores[k] * w for k, w in WEIGHTS.items()) / 10)


def apply_guardrails(ai_scores, rules):
    """The AI judges quality, but the rules cap the dimensions they can check for certain."""
    s = {k: float(ai_scores.get(k, rules["scores"][k])) for k in WEIGHTS}
    for k in ("accuracy", "consistency", "forward"):
        s[k] = min(s[k], rules["scores"][k])
    if any("Evasive" in m for _, m in rules["flags"]):
        s["directness"] = min(s["directness"], rules["scores"]["directness"])
    return s

Appending to engine.py


**Check:** three answers to the margin question (Q1):
- **Weak → 38/100**: unapproved "17%", repeated old promises, evasive phrases, absolute guarantees
- **Average → 66/100**: polite but vague; no figures and no admission of the miss (the typical "safe" executive answer)
- **Model → 97/100**: admits the miss, explains the 260 bps bridge, gives the revised outlook using only disclosed facts

In [ ]:
importlib.reload(engine)
q1 = data.QUESTIONS[0]
print("QUESTION:", q1["question"])
for name, ans in [("weak", data.DEMO_ANSWERS["weak"]),
                  ("average", data.DEMO_ANSWERS["average"]),
                  ("model", q1["model_answer"])]:
    r = engine.rule_check(ans, q1)
    print(f"\n{name.upper()} → {r['overall']}/100   {r['scores']}")
    for level, msg in r["flags"]:
        print("   ", {"error": "🔴", "warn": "🟠", "ok": "🟢"}[level], msg)

QUESTION: In May you guided to 50–100 bps of EBITDA margin expansion for FY27, and in August you said you were still comfortable with it. H1 margin is 14.4%, more than 200 bps below even the low end. Is the guidance withdrawn, and why should we trust the new number?

WEAK → 38/100   {'accuracy': 7, 'consistency': 2, 'directness': 0, 'forward': 4, 'clarity': 6}
    🔴 Figure '17' is not in the approved disclosure pack.
    🔴 Repeats a replaced promise as if still valid: 'Margin expansion of 50–100 bps / 16.8–17.3% margin'.
    🔴 Repeats a replaced promise as if still valid: 'Margin dip is temporary / comfortable with full-year margin guidance'.
    🟠 Does not acknowledge the gap versus earlier guidance. That sounds evasive.
    🔴 Evasive or dismissive phrase: 'i don't think there is any'.
    🔴 Evasive or dismissive phrase: 'doing fantastic'.
    🔴 Evasive or dismissive phrase: 'winning everywhere'.
    🔴 Absolute promise 'definitely'. Say 'we expect' and use the disclosed outlook.
    🔴

## 4. Readiness verdict and recommendations (`engine.py`, part 3)
Turns rehearsal scores into the **business recommendation**.

**Readiness index** = risk-weighted average of each topic's best rehearsal score. Missing a margin question (risk 25) counts five times more than missing a capex question (risk 4.7). Unrehearsed topics count as 0.

Recommendations are generated from the gaps and scores: what to pre-empt in prepared remarks, which old guidance to reset formally, who owns which questions, what to rehearse again, what to lead with, and which exhibits to prepare.

It also produces a **Q&A briefing pack** that leadership can read the night before the call.

In [ ]:
%%writefile -a engine.py


# engine.py — Part 3: readiness index and recommendations
def best_scores(history):
    """Best rehearsal score per topic (gap id)."""
    gap_of = {q["id"]: q["gap"] for q in D.QUESTIONS}
    best = {}
    for h in history:
        g = gap_of.get(h["qid"], h.get("gap"))
        best[g] = max(best.get(g, 0), h["overall"])
    return best


def readiness(gaps, history):
    best = best_scores(history)
    rows, num, den = [], 0, 0
    for g in gaps:
        sc = best.get(g["id"])
        num += g["risk"] * (sc or 0)
        den += g["risk"] * 100
        state = ("Not rehearsed" if sc is None else "Ready" if sc >= 75 else "Needs work" if sc >= 55 else "Not ready")
        rows.append({"Topic": g["topic"], "Risk": g["risk"], "RAG": g["rag"], "Owner": g["owner"],
                     "Best score": sc, "Readiness": state})
    index = round(100 * num / den) if den else 100
    verdict = "Ready" if index >= 75 else "Nearly ready" if index >= 55 else "Not ready"
    return index, verdict, rows


def recommendations(gaps, history):
    best = best_scores(history)
    red = [g for g in gaps if g["rag"] == "Red"]
    recs = []
    if red:
        recs.append(("Pre-empt in prepared remarks",
                     f"Address {', '.join(g['topic'] for g in red)} in the CEO/CFO opening remarks instead of waiting "
                     f"for Q&A. Lead with the revised outlook and the bridge: {red[0]['key_fact']}."))
    recs.append(("Formally reset old guidance",
                 "State clearly that these promises are replaced, and make sure no one repeats them: "
                 + "; ".join(c for c, _ in D.SUPERSEDED[:3]) + "."))
    owners = {}
    for g in gaps:
        if g["risk"] >= 7:
            owners.setdefault(g["owner"], []).append(g["topic"])
    recs.append(("Assign question owners", " · ".join(f"**{o}**: {', '.join(t)}" for o, t in owners.items())))
    weak = [g for g in gaps if g["risk"] >= 7 and best.get(g["id"], 0) < 75]
    if weak:
        recs.append(("Rehearse again before the call",
                     ", ".join(f"{g['topic']} ({'not rehearsed' if g['id'] not in best else 'best ' + str(best[g['id']])})"
                               for g in weak) + ". Target 75+ on every red/amber topic."))
    good = [g for g in gaps if g["status"] != "Missed"]
    if good:
        recs.append(("Lead with what went right", "Open with " + "; ".join(g["key_fact"] for g in good) + "."))
    recs.append(("Prepare exhibits", "Margin bridge (260 bps), receivables bridge, net debt walk to March 2027, "
                                     "and a revised-vs-original guidance table."))
    return recs


def briefing_pack(gaps, history):
    """Markdown Q&A brief leadership can read the night before the call."""
    gm = {g["id"]: g for g in gaps}
    best = {}
    for h in history:
        best[h["qid"]] = max(best.get(h["qid"], 0), h["overall"])
    out = [f"# Q&A briefing pack: {D.COMPANY_NAME}, {D.COMPANY['quarter']}", "_Synthetic data, for rehearsal only._", ""]
    for q in sorted(D.QUESTIONS, key=lambda q: -gm[q["gap"]]["risk"]):
        g = gm[q["gap"]]
        out += [f"## {q['id']}. {q['question']}",
                f"- Asked by: {q['persona']} · Owner: {g['owner']} · Risk: {g['risk']} ({g['rag']})",
                f"- Key facts: {g['key_fact']}",
                f"- Rehearsal score: {best.get(q['id'], 'not rehearsed')}",
                "", f"**Recommended answer:** {q['model_answer']}", "",
                f"**Likely follow-up:** {q['follow_up']}", ""]
    return "\n".join(out)

Appending to engine.py


**Check:** a sample rehearsal round (weak answer on Q1, average on Q2, strong on Q3, Q4 and Q9) should give a **readiness index of 48/100 → Not ready**.

Why 48? Total risk = 106.7. Points earned = margin 25 × 67 + FCF 22.5 × 100 + receivables 12 × 100 = 5,125. 5,125 ÷ 10,670 = 48%. Digital margin, Consumer and net debt were never rehearsed, so the CFO is not ready even though three answers scored 100.

In [ ]:
importlib.reload(engine)
gaps = engine.detect_gaps()
qs = {q["id"]: q for q in data.QUESTIONS}

# 5 practice attempts: weak on Q1, average on Q2, strong on Q3, Q4, Q9
attempts = [("Q1", data.DEMO_ANSWERS["weak"]), ("Q2", data.DEMO_ANSWERS["average"]),
            ("Q3", qs["Q3"]["model_answer"]), ("Q4", qs["Q4"]["model_answer"]), ("Q9", qs["Q9"]["model_answer"])]
history = [{"qid": qid, "overall": engine.rule_check(ans, qs[qid])["overall"]} for qid, ans in attempts]

index, verdict, rows = engine.readiness(gaps, history)
print(f"READINESS INDEX: {index}/100 → {verdict}\n")
display(pd.DataFrame(rows))
print("\nRECOMMENDATIONS")
for i, (title, body) in enumerate(engine.recommendations(gaps, history), 1):
    print(f"{i}. {title}: {body}\n")

READINESS INDEX: 48/100 → Not ready



,Topic,Risk,RAG,Owner,Best score,Readiness
0,EBITDA margin vs guidance,25.0,Red,CFO,67.0,Needs work
1,Free cash flow,22.5,Red,CFO,100.0,Ready
2,Digital segment margin,20.0,Red,Head of Digital,NaN,Not rehearsed
3,Working capital / receivables,12.0,Amber,CFO,100.0,Ready
4,Consumer segment turnaround,12.0,Amber,CEO,NaN,Not rehearsed
5,Net debt and balance sheet,10.5,Amber,CFO,NaN,Not rehearsed
6,Capex run-rate,4.7,Green,CFO,NaN,Not rehearsed
7,Revenue growth,0.0,Green,CEO,94.0,Ready
8,Digital growth,0.0,Green,Head of Digital,NaN,Not rehearsed



RECOMMENDATIONS
1. Pre-empt in prepared remarks: Address EBITDA margin vs guidance, Free cash flow, Digital segment margin in the CEO/CFO opening remarks instead of waiting for Q&A. Lead with the revised outlook and the bridge: Revised margin outlook 14.5–15.5%; 260 bps bridge (120 Digital / 100 Consumer / 40 Industrial).

2. Formally reset old guidance: State clearly that these promises are replaced, and make sure no one repeats them: Margin expansion of 50–100 bps / 16.8–17.3% margin; FCF positive every quarter / FY27 FCF over ₹600 crore; Consumer back to growth by Q2.

3. Assign question owners: **CFO**: EBITDA margin vs guidance, Free cash flow, Working capital / receivables, Net debt and balance sheet · **Head of Digital**: Digital segment margin · **CEO**: Consumer segment turnaround

4. Rehearse again before the call: EBITDA margin vs guidance (best 67), Digital segment margin (not rehearsed), Consumer segment turnaround (not rehearsed), Net debt and balance sheet (not rehearse

## 5. Gemini AI (`llm.py`)
Gemini adds what rules can't: qualitative feedback, an improved answer written only from disclosed facts, a realistic follow-up question, and fresh investor questions.

**Reliability built in:**
- Tries three Gemini models in order, and retries once if Google is busy (errors 429/503)
- If every call fails, the app falls back to rule-based scoring, so the demo never breaks

The API key is read from Colab Secrets (never written in the code) and saved to `.streamlit/secrets.toml` for the app. **Never upload that file to GitHub.**

In [ ]:
from google.colab import userdata
GEMINI_KEY = userdata.get("GEMINI_API_KEY")

# Save it where Streamlit can read it
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/secrets.toml", "w") as f:
    f.write(f'GEMINI_API_KEY = "{GEMINI_KEY}"\n')
print("Key loaded:", GEMINI_KEY[:6] + "…")

Key loaded: AQ.Ab8…


In [12]:
%%writefile llm.py
# llm.py — Gemini: scores answers and writes new questions. Returns None on any failure,
# so the app falls back to the rule-based guardrails.
import json
import time
from google import genai
from google.genai import types
import data as D

MODELS = ["gemini-flash-latest", "gemini-2.5-flash", "gemini-flash-lite-latest"]


def context():
    t = "\n".join(f"- [{i}] {call}, {who}: \"{quote}\"" for i, call, who, quote in D.TRANSCRIPTS)
    d = "\n".join(f"- {x}" for x in D.DISCLOSURES)
    s = "\n".join(f"- {c}" for c, _ in D.SUPERSEDED)
    return (f"COMPANY: {D.COMPANY_NAME}, {D.COMPANY['quarter']}. Figures in ₹ crore.\n\n"
            f"APPROVED DISCLOSURE PACK (the ONLY facts management may state):\n{d}\n\n"
            f"EARLIER MANAGEMENT STATEMENTS (investors compare against these):\n{t}\n\n"
            f"REPLACED PROMISES (must not be repeated as if still valid):\n{s}\n")


class Gemini:
    def __init__(self, api_key):
        self.ok = bool(api_key)
        self.client = genai.Client(api_key=api_key) if self.ok else None
        self.error, self.model = None, None

    def ask_json(self, prompt, temperature=0.2):
        if not self.ok:
            return None
        errors = []
        for m in MODELS:                 # try the newest model first, fall back if unavailable
            for attempt in range(2):     # retry once if Google is busy or rate-limiting
                try:
                    r = self.client.models.generate_content(
                        model=m, contents=prompt,
                        config=types.GenerateContentConfig(
                            temperature=temperature, response_mime_type="application/json",
                            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True)))
                    self.model = m
                    return json.loads(r.text)
                except Exception as e:
                    msg = str(e)
                    errors.append(f"{m}: {msg[:150]}")
                    if attempt == 0 and ("429" in msg or "503" in msg):
                        time.sleep(4)
                        continue
                    break
        self.error = " | ".join(errors)
        return None

    def score_answer(self, q, gap, answer):
        prompt = f"""You are an investor-relations coach preparing a listed Indian company's leadership for
its earnings call. Judge the executive's answer strictly against the disclosure pack and earlier statements.

{context()}
QUESTION (from {q['persona']}): {q['question']}
RELATED PROMISE: {gap['commitment']} ({gap['source']}); actual = {gap['actual']} {gap['unit']}.

EXECUTIVE'S ANSWER:
\"\"\"{answer}\"\"\"

Score 0-10 each:
- accuracy: every figure matches the disclosure pack; no new numbers
- consistency: no contradiction or repetition of replaced promises; admits changes honestly
- directness: answers the actual question and admits the gap
- forward: forward-looking statements hedged ("we expect"), only the disclosed outlook, no promises
- clarity: concise, structured, 30-75 seconds spoken, no filler

Return JSON only:
{{"scores": {{"accuracy": 0, "consistency": 0, "directness": 0, "forward": 0, "clarity": 0}},
  "strengths": ["..."], "issues": ["..."],
  "improved_answer": "80-140 words using ONLY disclosure-pack facts",
  "follow_up": "the toughest follow-up an analyst would ask next"}}"""
        return self.ask_json(prompt)

    def new_questions(self, gaps, n=5):
        g = "\n".join(f"- {x['topic']}: promised {x['commitment']}; actual {x['actual']} {x['unit']} (risk {x['risk']})"
                      for x in gaps[:6])
        prompt = f"""{context()}
GAPS FOUND (highest risk first):
{g}

Write {n} NEW tough questions that sell-side analysts, institutional investors or credit analysts will
likely ask on this call. Each must cite specific numbers or earlier quotes. Return JSON only:
{{"questions": [{{"persona": "...", "topic": "...", "question": "...", "why": "..."}}]}}"""
        out = self.ask_json(prompt, temperature=0.7)
        return out.get("questions") if out else None

Writing llm.py


**Check:** Gemini scores the weak answer, then the guardrails cap it. Expect a very low final score, a list of issues, an improved answer that uses the revised 14.5–15.5% outlook, and a tough follow-up question. AI scores vary slightly between runs.

In [13]:
import llm
importlib.reload(llm)
g = llm.Gemini(GEMINI_KEY)
q1, weak = data.QUESTIONS[0], data.DEMO_ANSWERS["weak"]

res = g.score_answer(q1, gaps[0], weak)
if res is None:
    print("❌ All models failed:\n", g.error)
else:
    rules = engine.rule_check(weak, q1)
    final = engine.apply_guardrails(res["scores"], rules)
    print("Model used:     ", g.model)
    print("Gemini scores:  ", res["scores"])
    print("Rule scores:    ", rules["scores"])
    print("Final (capped): ", final, "→", engine.overall(final), "/100")
    print("\nIssues Gemini found:", *res["issues"], sep="\n - ")
    print("\nImproved answer:\n", res["improved_answer"])
    print("\nLikely follow-up:", res["follow_up"])

Model used:      gemini-flash-lite-latest
Gemini scores:   {'accuracy': 2, 'consistency': 0, 'directness': 0, 'forward': 0, 'clarity': 2}
Rule scores:     {'accuracy': 7, 'consistency': 2, 'directness': 0, 'forward': 4, 'clarity': 6}
Final (capped):  {'accuracy': 2.0, 'consistency': 0.0, 'directness': 0.0, 'forward': 0.0, 'clarity': 2.0} → 8 /100

Issues Gemini found:
 - Repeatedly violates the replaced promises by claiming the margin dip is temporary and promising a return to 17% margins by year-end.
 - Completely fails to acknowledge or directly answer the question about the missed previous guidance.
 - Fails to use any factual metrics from the approved disclosure pack (such as the revised guidance of 14.5-15.5% or the margin bridge breakdown).
 - Uses forbidden absolute language ('no doubt about it') instead of appropriate hedging.

Improved answer:
 We have revised our full-year EBITDA margin outlook to 14.5 to 15.5 percent. The gap in H1 reflects transition costs on three large Di

## 6. Streamlit dashboard (`app.py`)
The interactive app, built in four parts:
- **A.** Setup, sidebar (company info, Gemini status, demo controls) and header
- **B.** ① Results (KPIs, trend charts, segments, margin bridge) and ② Gap detector (risk table and chart, past quotes)
- **C.** ③ Likely questions (ranked bank + Gemini-generated questions) and ④ Practise & score (the core rehearsal loop)
- **D.** ⑤ Readiness (index, topic table, recommendations, briefing pack download) and Data & assumptions

In [14]:
%%writefile app.py
# app.py — Part A: setup, sidebar, header
import os
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import streamlit as st
import data as D
import engine as E
from llm import Gemini

st.set_page_config(page_title="Earnings Call Readiness Coach", page_icon="🎙️", layout="wide")
RAG = {"Red": "#D64545", "Amber": "#E8A33D", "Green": "#3A9D5D"}
NAVY = "#1F3A68"
st.markdown("""<style>
.card {border:1px solid rgba(128,128,128,.25); border-radius:10px; padding:14px 16px; margin-bottom:10px; text-align:center}
.qcard {border-left:5px solid #1F3A68; background:rgba(31,58,104,.06); border-radius:8px; padding:14px 18px; margin:6px 0 12px}
.pill {display:inline-block; padding:2px 10px; border-radius:999px; font-size:.78rem; font-weight:600; color:white; margin-right:6px}
.big {font-size:3rem; font-weight:700; line-height:1}
.muted {opacity:.7; font-size:.85rem}
</style>""", unsafe_allow_html=True)


def pill(text, color):
    return f'<span class="pill" style="background:{color}">{text}</span>'


def score_color(s):
    return "#3A9D5D" if s >= 75 else "#E8A33D" if s >= 55 else "#D64545"


gaps = E.detect_gaps()
gap_by_id = {g["id"]: g for g in gaps}
ss = st.session_state
ss.setdefault("history", [])
ss.setdefault("extra_questions", [])
ss.setdefault("ai_questions", [])
ss.setdefault("answer", "")
ss.setdefault("result", None)


def all_questions():
    return sorted(D.QUESTIONS + ss.extra_questions, key=lambda q: -gap_by_id[q["gap"]]["risk"])


def get_key():
    try:
        return st.secrets.get("GEMINI_API_KEY", "")
    except Exception:
        return os.environ.get("GEMINI_API_KEY", "")


with st.sidebar:
    st.markdown(f"### {D.COMPANY_NAME}")
    st.caption(f"{D.COMPANY['ticker']} · {D.COMPANY['quarter']} · call in {D.COMPANY['call_date']}")
    st.caption(D.COMPANY["description"])
    st.divider()
    key = st.text_input("Gemini API key", value=get_key(), type="password")
    llm = Gemini(key)
    st.caption("🟢 Gemini connected" if llm.ok else "⚪ Offline mode: rule-based scoring")
    st.divider()
    if st.button("Load sample rehearsal round", width="stretch"):
        qs = {q["id"]: q for q in D.QUESTIONS}
        plan = [("Q1", D.DEMO_ANSWERS["weak"]), ("Q2", D.DEMO_ANSWERS["average"]),
                ("Q3", qs["Q3"]["model_answer"]), ("Q4", qs["Q4"]["model_answer"]), ("Q9", qs["Q9"]["model_answer"])]
        ss.history = [{"qid": qid, "overall": E.rule_check(a, qs[qid])["overall"], "engine": "demo",
                       "time": datetime.now().strftime("%H:%M")} for qid, a in plan]
        st.success("Loaded 5 sample attempts.")
    if st.button("Reset rehearsal", width="stretch"):
        ss.history, ss.result = [], None
    st.caption(f"Attempts logged: {len(ss.history)}")
    st.caption("Synthetic data, for demonstration only.")

st.title("🎙️ Earnings Call Readiness Coach")
st.markdown(f"Is **{D.COMPANY_NAME}**'s leadership ready for tough investor questions? "
            "Results go in; a readiness verdict and a briefing pack come out.")
tabs = st.tabs(["① Results", "② Gap detector", "③ Likely questions", "④ Practise & score", "⑤ Readiness",
                "Data & assumptions"])

Writing app.py


In [15]:
%%writefile -a app.py

# app.py — Part B: Results and Gap detector tabs
q = pd.DataFrame(D.QUARTERLY)
q["fcf"] = q.cfo - q.capex
q["margin"] = q.ebitda / q.revenue * 100
cur, ly = q.iloc[-1], q.iloc[1]          # Q2 FY27 and Q2 FY26

with tabs[0]:
    st.subheader(f"{D.COMPANY['quarter']} at a glance")
    c = st.columns(5)
    c[0].metric("Revenue (₹ cr)", f"{cur.revenue:,}", f"{(cur.revenue / ly.revenue - 1) * 100:+.1f}% YoY")
    c[1].metric("EBITDA margin", f"{cur.margin:.1f}%", f"{(cur.margin - ly.margin) * 100:+.0f} bps YoY")
    c[2].metric("PAT (₹ cr)", f"{cur.pat:,}", f"{(cur.pat / ly.pat - 1) * 100:+.1f}% YoY")
    c[3].metric("Free cash flow (₹ cr)", f"{cur.fcf:,}", f"{cur.fcf - ly.fcf:+,} vs last year")
    c[4].metric("Net debt (₹ cr)", f"{cur.net_debt:,.0f}", f"{cur.net_debt - 610:+,.0f} since Mar-26", delta_color="inverse")
    st.info("**What investors will see:** revenue is growing in line with guidance, but margins, cash flow and net debt "
            "have moved the wrong way, and several earlier management promises are now broken.")

    left, right = st.columns(2)
    fig = go.Figure()
    fig.add_bar(x=q.quarter, y=q.revenue, name="Revenue (₹ cr)", marker_color="#9FB3D1")
    fig.add_scatter(x=q.quarter, y=q.margin, name="EBITDA margin %", yaxis="y2", mode="lines+markers+text",
                    text=[f"{v:.1f}%" for v in q.margin], textposition="top center", line=dict(color=NAVY, width=3))
    fig.update_layout(title="Revenue up, margin down", height=340, legend=dict(orientation="h", y=-0.15),
                      yaxis2=dict(overlaying="y", side="right", range=[12, 18], dtick=1, ticksuffix="%", showgrid=False))
    left.plotly_chart(fig, width="stretch")
    fig = go.Figure(go.Bar(x=q.quarter, y=q.fcf, text=q.fcf, textposition="outside",
                           marker_color=["#D64545" if v < 0 else "#3A9D5D" for v in q.fcf]))
    fig.update_layout(title="Free cash flow (₹ cr) has turned negative", height=340, yaxis=dict(range=[-150, 280]))
    right.plotly_chart(fig, width="stretch")

    left, right = st.columns([1.35, 1])
    s = pd.DataFrame(D.SEGMENTS)
    left.markdown("**Segment performance (Q2 FY27 vs Q2 FY26)**")
    left.dataframe(pd.DataFrame({
        "Segment": s.segment, "Revenue (₹ cr)": s.rev_q2fy27,
        "YoY growth": [f"{(a / b - 1) * 100:+.1f}%" for a, b in zip(s.rev_q2fy27, s.rev_q2fy26)],
        "EBIT margin": [f"{m:.1f}%" for m in s.margin_q2fy27],
        "Δ margin": [f"{(a - b) * 100:+.0f} bps" for a, b in zip(s.margin_q2fy27, s.margin_q2fy26)]}),
        hide_index=True, width="stretch")
    left.caption("Digital: fast growth, margins diluted · Industrial: steady · Consumer: shrinking, near break-even")
    fig = go.Figure(go.Waterfall(
        x=[b[0] for b in D.MARGIN_BRIDGE], y=[b[1] for b in D.MARGIN_BRIDGE], measure=[b[2] for b in D.MARGIN_BRIDGE],
        text=[f"{b[1]:.1f}%" if b[2] != "relative" else f"{b[1] * 100:.0f} bps" for b in D.MARGIN_BRIDGE],
        textposition="outside", decreasing=dict(marker_color="#D64545"), totals=dict(marker_color=NAVY)))
    fig.update_layout(title="EBITDA margin bridge (YoY)", height=320, yaxis=dict(range=[12, 17.5]))
    right.plotly_chart(fig, width="stretch")

with tabs[1]:
    st.subheader("Where results break earlier promises")
    st.caption("Risk = gap severity (0–5) × investor sensitivity (1–5). Red ≥ 15 · Amber 7–15 · Green < 7")
    c = st.columns(3)
    c[0].metric("Promises tracked", len(gaps))
    c[1].metric("Missed", sum(g["status"] == "Missed" for g in gaps))
    c[2].metric("High-risk (red) topics", sum(g["rag"] == "Red" for g in gaps))

    def fmt(g):
        a = g["actual"]
        if g["unit"] == "₹ cr":
            return f"-₹{-a:,g} cr" if a < 0 else f"₹{a:,g} cr"
        return f"{a:g}%" if g["unit"] == "%" else f"{a:g} {g['unit']}"

    table = pd.DataFrame([{"Topic": g["topic"], "Earlier promise": g["commitment"], "Said in": g["source"],
                           "Actual": fmt(g), "Severity": g["severity"], "Sens.": g["sensitivity"],
                           "Risk": g["risk"], "RAG": g["rag"]} for g in gaps])
    st.dataframe(table.style.map(lambda v: f"background-color:{RAG[v]};color:white;font-weight:600" if v in RAG else "",
                                 subset=["RAG"]).format({"Risk": "{:.1f}", "Severity": "{:.1f}"}),
                 hide_index=True, width="stretch", height=370)
    left, right = st.columns([1.3, 1])
    rg = gaps[::-1]
    fig = go.Figure(go.Bar(y=[g["topic"] for g in rg], x=[g["risk"] for g in rg], orientation="h",
                           marker_color=[RAG[g["rag"]] for g in rg], text=[g["risk"] for g in rg], textposition="outside"))
    fig.update_layout(title="Risk score by topic (max 25)", height=330, xaxis=dict(range=[0, 30]))
    left.plotly_chart(fig, width="stretch")
    right.markdown("**What management said before**")
    for i, call, who, quote in D.TRANSCRIPTS:
        right.markdown(f"<div class='muted'><b>[{i}] {who}, {call}:</b> “{quote}”</div>", unsafe_allow_html=True)

Appending to app.py


In [16]:
%%writefile -a app.py

# app.py — Part C: Likely questions and Practise tabs
with tabs[2]:
    st.subheader("The questions investors are most likely to ask")
    st.caption("Ranked by the risk of the broken promise behind each question.")
    for qq in all_questions():
        g = gap_by_id[qq["gap"]]
        with st.expander(f"{qq['id']} · [{g['rag']}] {qq['question'][:110]}…"):
            st.markdown(pill(f"{g['rag']} risk {g['risk']}", RAG[g["rag"]]) + pill("Owner: " + g["owner"], NAVY),
                        unsafe_allow_html=True)
            st.markdown(f"**Asked by:** {qq['persona']}\n\n> {qq['question']}\n\n**Why they'll ask:** {qq['why']}\n\n"
                        f"**Facts to anchor on:** {g['key_fact']}")
    st.divider()
    if st.button("✨ Generate 5 new questions with Gemini", disabled=not llm.ok):
        with st.spinner("Thinking like a sell-side analyst…"):
            ss.ai_questions = llm.new_questions(gaps) or []
        if not ss.ai_questions:
            st.error(f"Gemini call failed: {llm.error}")
    for i, aq in enumerate(ss.ai_questions):
        st.markdown(f"**{aq.get('persona', 'Analyst')}** · {aq.get('question', '')}  \n_{aq.get('why', '')}_")
        if st.button("Add to practice list", key=f"add{i}"):
            text = (aq.get("topic", "") + " " + aq.get("question", "")).lower()
            gid = next((gid for words, gid in [(("cash", "fcf"), "fcf"), (("receivable", "working capital"), "receivables"),
                        (("consumer", "oem"), "consumer"), (("debt", "dividend"), "debt"), (("capex",), "capex"),
                        (("digital",), "digital")] if any(w in text for w in words)), "margin")
            ss.extra_questions.append(dict(id=f"AI{len(ss.extra_questions) + 1}", gap=gid, persona=aq.get("persona", "Analyst"),
                                           question=aq.get("question", ""), why=aq.get("why", ""),
                                           keywords=[gid, "guidance"], model_answer="", follow_up=""))
            st.success("Added. Pick it in the Practise tab.")

with tabs[3]:
    st.subheader("Practise an answer and get scored")
    options = all_questions()
    qid = st.selectbox("Pick a question (highest risk first)", [x["id"] for x in options],
                       format_func=lambda i: next(f"{i} · [{gap_by_id[x['gap']]['rag']}] {x['question'][:90]}…"
                                                  for x in options if x["id"] == i))
    qq = next(x for x in options if x["id"] == qid)
    g = gap_by_id[qq["gap"]]
    st.markdown(f'<div class="qcard"><div class="muted">🎤 {qq["persona"]} asks:</div>'
                f'<div style="font-size:1.12rem;margin-top:4px">“{qq["question"]}”</div></div>', unsafe_allow_html=True)
    st.caption(f"Owner: {g['owner']} · Promise at risk: {g['commitment']} · Facts to use: {g['key_fact']}")

    b = st.columns([1.5, 1.5, 0.8, 2.2])
    if b[0].button("Demo: weak answer"):
        ss.answer = D.DEMO_ANSWERS["weak"]
    if b[1].button("Demo: average answer"):
        ss.answer = D.DEMO_ANSWERS["average"]
    if b[2].button("Clear"):
        ss.answer = ""
    answer = st.text_area("Your answer (as you would say it on the call)", key="answer", height=150)

    if st.button("🎯 Score my answer", type="primary", disabled=not answer.strip()):
        rules = E.rule_check(answer, qq)
        ai = None
        if llm.ok:
            with st.spinner("Gemini is checking the answer against the disclosure pack…"):
                ai = llm.score_answer(qq, g, answer)
        if ai and "scores" in ai:
            scores, engine = E.apply_guardrails(ai["scores"], rules), f"Gemini ({llm.model}) + guardrails"
        else:
            scores, engine = rules["scores"], "Rule-based guardrails (offline)"
            if llm.ok:
                st.warning(f"Gemini unavailable, used rules only. {llm.error}")
        total = E.overall(scores)
        ss.result = dict(qid=qid, scores=scores, overall=total, rules=rules, ai=ai, engine=engine)
        ss.history.append(dict(qid=qid, gap=qq["gap"], overall=total, engine=engine, time=datetime.now().strftime("%H:%M")))

    res = ss.result
    if res and res["qid"] == qid:
        st.divider()
        left, right = st.columns([1, 2])
        col = score_color(res["overall"])
        verdict = "Call-ready" if res["overall"] >= 75 else "Needs work" if res["overall"] >= 55 else "Not ready"
        left.markdown(f'<div class="card"><div class="muted">Answer score</div><div class="big" style="color:{col}">'
                      f'{res["overall"]}</div><div class="muted">out of 100</div>{pill(verdict, col)}</div>',
                      unsafe_allow_html=True)
        left.caption(f"Scored by: {res['engine']}")
        for k, label in E.LABELS.items():
            right.markdown(f"{label} ({E.WEIGHTS[k]}%): **{res['scores'][k]:.1f}/10**")
            right.progress(min(1.0, res["scores"][k] / 10))

        left, right = st.columns(2)
        left.markdown("**🛡️ Disclosure guardrails** (rule-based, always on)")
        for level, msg in res["rules"]["flags"]:
            left.markdown({"error": "🔴 ", "warn": "🟠 ", "ok": "🟢 "}[level] + msg)
        ai = res["ai"] or {}
        right.markdown("**🤖 Coach feedback (Gemini)**")
        if not ai:
            right.caption("Connect Gemini for qualitative feedback. The guardrails still apply.")
        for s_ in ai.get("strengths", [])[:3]:
            right.markdown(f"✅ {s_}")
        for s_ in ai.get("issues", [])[:4]:
            right.markdown(f"⚠️ {s_}")

        better = ai.get("improved_answer") or qq.get("model_answer")
        if better:
            st.markdown("**✍️ Stronger answer (uses only disclosed facts)**")
            st.success(better)
        follow = ai.get("follow_up") or qq.get("follow_up")
        if follow:
            st.markdown(f"**🔁 Expect this follow-up:** _{follow}_")
            if st.button("Practise this follow-up next"):
                ss.extra_questions.append(dict(id=f"FU{len(ss.extra_questions) + 1}", gap=qq["gap"],
                                               persona=qq["persona"] + " (follow-up)", question=follow,
                                               why=f"Follow-up to {qid}.", keywords=qq["keywords"],
                                               model_answer="", follow_up=""))
                st.success("Added to the practice list.")

Appending to app.py


In [17]:
%%writefile -a app.py

# app.py — Part D: Readiness and Data tabs
with tabs[4]:
    st.subheader("Is leadership ready for the call?")
    index, verdict, rows = E.readiness(gaps, ss.history)
    col = score_color(index)
    left, right = st.columns([1, 2.2])
    left.markdown(f'<div class="card"><div class="muted">Readiness index</div><div class="big" style="color:{col}">'
                  f'{index}</div><div class="muted">risk-weighted, out of 100</div>{pill(verdict, col)}</div>',
                  unsafe_allow_html=True)
    left.caption(f"Based on {len(ss.history)} rehearsal attempt(s). Unrehearsed topics count as 0.")
    if not ss.history:
        left.info("No rehearsals yet. Practise in tab ④ or click **Load sample rehearsal round** in the sidebar.")
    r = pd.DataFrame(rows)
    r["Risk"] = r["Risk"].map(lambda v: f"{v:.1f}")
    r["Best score"] = r["Best score"].map(lambda v: "–" if pd.isna(v) else f"{v:.0f}")
    state_color = {"Ready": "#3A9D5D", "Needs work": "#E8A33D", "Not ready": "#D64545", "Not rehearsed": "#888"}
    right.dataframe(r.style.map(lambda v: f"background-color:{RAG[v]};color:white" if v in RAG else "", subset=["RAG"])
                    .map(lambda v: f"color:{state_color.get(v, '')};font-weight:600", subset=["Readiness"]),
                    hide_index=True, width="stretch", height=360)

    st.markdown("### 📋 Recommendation to leadership")
    for i, (title, body) in enumerate(E.recommendations(gaps, ss.history), 1):
        st.markdown(f"**{i}. {title}.** {body}")
    st.download_button("⬇️ Download Q&A briefing pack (.md)", E.briefing_pack(gaps, ss.history),
                       file_name="qa_briefing_pack.md")

with tabs[5]:
    st.subheader("Inputs and assumptions")
    for a in D.ASSUMPTIONS:
        st.markdown(f"- {a}")
    left, right = st.columns(2)
    left.markdown("**Approved Q2 FY27 disclosure pack (source of truth)**")
    for d in D.DISCLOSURES:
        left.markdown(f"- {d}")
    right.markdown("**Earlier promises (previous call transcripts)**")
    for i, call, who, quote in D.TRANSCRIPTS:
        right.markdown(f"- **[{i}] {who}, {call}:** “{quote}”")
    st.markdown("**Quarterly financials (₹ crore)**")
    st.dataframe(q.round(1), hide_index=True, width="stretch")
    st.markdown("**Segments**")
    st.dataframe(pd.DataFrame(D.SEGMENTS), hide_index=True, width="stretch")

Appending to app.py


## 7. Launch the app
Colab can't open a web page directly, so Streamlit runs in the background and a free **Cloudflare tunnel** gives it a public link.

- Click the `…trycloudflare.com` link printed below
- The link only works while this Colab session is running (the permanent link is created on Streamlit Cloud, see Section 8)
- After changing `data.py`, `engine.py` or `llm.py`, run the **restart** cell. The link stays the same

In [18]:
%%writefile .streamlit/config.toml
[theme]
base = "light"
primaryColor = "#1F3A68"

[server]
headless = true

Writing .streamlit/config.toml


In [19]:
![ -f cloudflared ] || wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pkill -f "[s]treamlit run" ; pkill -f "[c]loudflared tunnel" ; sleep 1
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &
!nohup ./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &

import time, re
time.sleep(12)
urls = re.findall(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
print("👉 Open your app:", urls[0] if urls else "not ready yet, wait 10 seconds and re-run just these last 4 lines")

👉 Open your app: https://acre-vernon-milwaukee-exhibitions.trycloudflare.com


**Restart Streamlit only** (keeps the same link). Use this after editing `data.py`, `engine.py` or `llm.py`.
If the app shows an error, run `!tail -20 streamlit.log` to see why.

In [20]:
!pkill -f "[s]treamlit run" ; sleep 2
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &
import time; time.sleep(6)
print("Restarted. Refresh the app tab.")

Restarted. Refresh the app tab.


## 8. Export for GitHub and Streamlit Cloud
Creates `requirements.txt` (libraries Streamlit Cloud must install) and `README.md` (the repo's front page), then downloads the app files as a zip.

🔒 `.streamlit/secrets.toml` (your API key) is **deliberately excluded**. On Streamlit Cloud, the key goes in *Advanced settings → Secrets* instead.

**Deploy:** upload the files + this notebook to GitHub → share.streamlit.io → *Create app* → choose the repo and `app.py` → add `GEMINI_API_KEY` in Secrets → *Deploy*.

In [21]:
%%writefile requirements.txt
streamlit>=1.50
pandas
plotly
google-genai

Writing requirements.txt


In [22]:
%%writefile README.md
# 🎙️ Earnings Call Readiness Coach — Barenya Pvt. Ltd.

**🔗 Live app:** PASTE_YOUR_STREAMLIT_LINK_HERE

A prototype that helps a listed company's leadership rehearse for a tough quarterly earnings call:
revenue grew, but margins fell, cash flow weakened and earlier management promises were missed.

## How it works
1. **Results**: Q2 FY27 KPIs, trends, segment performance and the margin bridge
2. **Gap detector**: compares 9 earlier management promises with actual results. Risk = severity × investor sensitivity
3. **Likely questions**: 10 tough investor questions ranked by risk (plus fresh ones from Gemini)
4. **Practise & score**: Gemini scores an executive's answer; fixed rules flag unapproved figures, repeated old promises, absolute promises and evasion, and cap the AI score
5. **Readiness**: risk-weighted readiness index, a 6-point recommendation to leadership and a downloadable Q&A briefing pack

## Data and assumptions
All data is **synthetic** (fictional company, ₹ crore). Full assumptions are in the app's *Data & assumptions* tab and in `data.py`.

## Key limitation
It judges *what* is said, not tone or delivery, and AI feedback must be reviewed by finance and legal before the call.

## Files
| File | Purpose |
|---|---|
| `data.py` | All inputs: financials, segments, past call transcripts, disclosure pack, promises, questions |
| `engine.py` | Rule-based logic: gap scoring, answer guardrails, readiness index, recommendations |
| `llm.py` | Gemini integration (answer scoring, question generation) |
| `app.py` | Streamlit dashboard |
| `Earnings_Call_Readiness_Coach.ipynb` | Colab notebook used to build and test everything step by step |

## Run locally
`pip install -r requirements.txt` then `streamlit run app.py`. Add `GEMINI_API_KEY` in the sidebar or in `.streamlit/secrets.toml`.
Without a key it runs in offline mode with rule-based scoring.

Writing README.md


In [23]:
!zip -q barenya_coach.zip app.py data.py engine.py llm.py requirements.txt README.md
from google.colab import files
files.download("barenya_coach.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9. Business value and limitation

**Business value**
- **Consistency:** every answer is grounded in the approved disclosure pack, which reduces the risk of selective disclosure or contradicting earlier statements
- **Focus:** leadership rehearses the highest-risk topics first, ranked by how far results missed each promise and how much investors care
- **Speed:** the finance/IR team gets a readiness verdict, question owners and a briefing pack in minutes instead of days of manual prep

**Key limitation**
The tool judges *what* is said, not *how* it is said (tone, confidence, body language), and AI feedback must still be reviewed by finance and legal before the call.

**Possible next steps:** voice input with speech-to-text for live rehearsal, loading real company filings and transcripts, and tracking readiness across quarters.